# Seasonal Agriculture Performance Analysis
**VOIS for Tech / AICTE Internship — Batch 1, 2026-27 | Major Project (Data Analytics)**

**Student:**_Arya Shukla_· **College:**_Dr.A.P.J.Abdul Kalam Technical University, Uttar Pradesh, Lucknow_· **AICTE STU ID:** _STU6830b928a14e51748023592_

---

## 1. Problem statement
Agricultural performance is shaped by the season: rainfall, temperature, soil moisture, pest
pressure, water use and market conditions all differ between **Kharif**, **Rabi** and **Zaid**.
Raw farm-level data does not say *how much* performance changes, *which* factors move with it,
or *where* the change hurts the farmer most.

This notebook analyses 4,000 farm records (28 variables, 8 states, 8 crops, 4 irrigation methods,
3 seasons) to quantify seasonal differences in **yield, resource efficiency and profitability**,
test whether those differences are statistically real, and turn them into recommendations.

## 2. Analytical questions
1. How large is the seasonal gap in yield, and does it survive after controlling for crop mix?
2. Which environmental conditions distinguish the three seasons?
3. How do economics — revenue, cost, profit, loss frequency — vary across seasons?
4. Does irrigation method change how badly a season hits a farm?
5. Which variables actually correlate with yield and profit, and which do not?
6. Are seasonal patterns uniform across states, or do some regions cope better?


## 3. Setup and data loading


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Season palette used throughout: Kharif (wet), Rabi (mild), Zaid (hot/dry)
KHARIF, RABI, ZAID = "#2C5F2D", "#97BC62", "#C96A2B"
SEASON_COLORS = {"Kharif": KHARIF, "Rabi": RABI, "Zaid": ZAID}
SEASON_ORDER = ["Kharif", "Rabi", "Zaid"]

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
sns.set_style("white")


In [ ]:
df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
print("Shape:", df.shape)
df.head()


## 4. Data understanding


In [ ]:
df.info()


In [ ]:
df.describe().T.round(2)


In [ ]:
for col in ["State", "District", "Crop", "Season", "Irrigation_Method"]:
    print(f"--- {col} ({df[col].nunique()} unique)")
    print(df[col].value_counts(), "\n")


### 4.1 Internal consistency check
Before trusting the derived columns, verify that the dataset's own arithmetic holds:
`Production = Yield x Area`, `Revenue = Production x Price`, `Profit = Revenue - Cost`.


In [ ]:
chk = pd.DataFrame({
    "production": np.isclose(df.Production_Tonnes,
                             df.Yield_Tonnes_Ha * df.Farm_Area_Hectares, rtol=0.02, equal_nan=True),
    "revenue": np.isclose(df.Revenue_INR,
                          df.Production_Tonnes * df.Market_Price_INR_Tonne, rtol=0.02, equal_nan=True),
    "profit": np.isclose(df.Profit_INR, df.Revenue_INR - df.Total_Cost_INR, rtol=0.01, equal_nan=True),
})
print(chk.mean().round(4))   # share of rows that are internally consistent


## 5. Data cleaning and preparation

Issues found and how they are handled:

| Issue | Decision |
|---|---|
| Missing `Rainfall_mm` (48) and `Soil_Moisture_pct` (40) | Median imputation **within Season x State** — rainfall is season- and region-specific, so a global median would flatten the very signal being studied |
| Missing `Yield_Tonnes_Ha` (32) | Median imputation **within Crop x Season** — yields differ by an order of magnitude between crops (sugarcane vs pulses) |
| Duplicates | Checked; none present |
| Season stored as unordered text | Converted to an ordered categorical so every table and plot follows Kharif → Rabi → Zaid |

**Engineered features**
- `Yield_Index` — yield divided by the crop's own mean. This removes the crop effect, so seasons can be compared fairly even if the crop mix differs.
- `Profit_per_ha` — profit scaled by farm area, so large and small farms are comparable.
- `Is_Loss` — flag for farms where profit is negative.


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Farm_IDs:", df.Farm_ID.duplicated().sum())
df.isna().sum()[lambda s: s > 0]


In [ ]:
for col in ["Rainfall_mm", "Soil_Moisture_pct"]:
    df[col] = df[col].fillna(df.groupby(["Season", "State"])[col].transform("median"))

df["Yield_Tonnes_Ha"] = df["Yield_Tonnes_Ha"].fillna(
    df.groupby(["Crop", "Season"])["Yield_Tonnes_Ha"].transform("median"))

df["Season"] = pd.Categorical(df["Season"], categories=SEASON_ORDER, ordered=True)

df["Profit_per_ha"] = df.Profit_INR / df.Farm_Area_Hectares
df["Is_Loss"] = df.Profit_INR < 0
df["Yield_Index"] = df.groupby("Crop").Yield_Tonnes_Ha.transform(lambda s: s / s.mean())

print("Remaining missing values:", int(df.isna().sum().sum()))
df[["Crop", "Season", "Yield_Tonnes_Ha", "Yield_Index", "Profit_per_ha", "Is_Loss"]].head()


### 5.1 Is the crop mix comparable across seasons?
If Kharif were dominated by high-yielding crops, any seasonal gap would be a composition
artefact. The crosstab below shows the crop mix is near-identical across seasons — which is
why the `Yield_Index` comparison that follows is meaningful.


In [ ]:
print(pd.crosstab(df.Season, df.Crop, normalize="index").round(3))
print()
print(pd.crosstab(df.Season, df.Irrigation_Method, normalize="index").round(3))


## 6. Q1 — How large is the seasonal gap?


In [ ]:
summary = df.groupby("Season", observed=True).agg(
    farms=("Farm_ID", "count"),
    yield_index=("Yield_Index", "mean"),
    yield_t_ha=("Yield_Tonnes_Ha", "mean"),
    profit_per_ha=("Profit_per_ha", "mean"),
    loss_share_pct=("Is_Loss", lambda s: 100 * s.mean()),
    water_eff=("Water_Efficiency_t_per_1000m3", "mean"),
).round(2)
summary


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

m = df.groupby("Season", observed=True).Yield_Index.mean() * 100
b = ax[0].bar(SEASON_ORDER, m[SEASON_ORDER], color=[SEASON_COLORS[s] for s in SEASON_ORDER], width=.6)
ax[0].bar_label(b, fmt="%.0f", padding=3, fontweight="bold")
ax[0].axhline(100, ls="--", lw=1, color="#888")
ax[0].set_title("Crop-normalised yield index")
ax[0].set_ylabel("Index (all-season average = 100)")
ax[0].set_ylim(0, 130)

l = df.groupby("Season", observed=True).Is_Loss.mean() * 100
b = ax[1].bar(SEASON_ORDER, l[SEASON_ORDER], color=[SEASON_COLORS[s] for s in SEASON_ORDER], width=.6)
ax[1].bar_label(b, fmt="%.1f%%", padding=3, fontweight="bold")
ax[1].set_title("Share of farms making a loss")
ax[1].set_ylabel("% of farms"); ax[1].set_ylim(0, 80)

plt.tight_layout(); plt.show()


**Observation.** Kharif runs ~11% above the all-season average yield, Rabi ~4% below and Zaid ~21%
below. Loss frequency moves in the same direction: 42.2% of Kharif farms are loss-making against
64.5% in Zaid.


### 6.1 Is the difference statistically significant?
Yield and profit are heavily skewed, so the non-parametric **Kruskal-Wallis** test is the primary
test; one-way **ANOVA** is reported alongside as a cross-check.


In [ ]:
def season_test(col):
    groups = [g.dropna().values for _, g in df.groupby("Season", observed=True)[col]]
    H, p_kw = st.kruskal(*groups)
    F, p_an = st.f_oneway(*groups)
    return {"variable": col, "kruskal_H": round(H, 1), "kruskal_p": f"{p_kw:.2e}",
            "anova_F": round(F, 1), "anova_p": f"{p_an:.2e}"}

pd.DataFrame([season_test(c) for c in
              ["Yield_Index", "Profit_per_ha", "Water_Efficiency_t_per_1000m3",
               "Disease_Pest_Risk_pct", "Rainfall_mm"]])


In [ ]:
# Is irrigation method distributed differently across seasons? (chi-square test of independence)
ct = pd.crosstab(df.Season, df.Irrigation_Method)
chi2, p, dof, _ = st.chi2_contingency(ct)
print(f"chi2 = {chi2:.2f}, dof = {dof}, p = {p:.3f}")
print("-> p > 0.05: irrigation mix is NOT significantly different across seasons,")
print("   so later irrigation comparisons are not confounded by season composition.")


**Result.** Every performance variable differs across seasons at p < 0.001, so the seasonal gap is
real and not sampling noise. Irrigation mix, by contrast, does not differ across seasons — useful,
because it means the irrigation comparison in section 9 is a like-for-like one.


## 7. Q2 — Does the gap hold within every crop?


In [ ]:
pivot = df.pivot_table(index="Crop", columns="Season", values="Yield_Index", observed=True)[SEASON_ORDER]
pivot = pivot.loc[df.groupby("Crop").Yield_Tonnes_Ha.mean().sort_values(ascending=False).index]

ax = pivot.plot(kind="barh", figsize=(9, 5), color=[KHARIF, RABI, ZAID], width=.78)
ax.axvline(1, ls="--", lw=1, color="#888")
ax.set_title("Kharif leads in every single crop")
ax.set_xlabel("Yield index (crop average = 1.0)"); ax.set_ylabel("")
ax.legend(title="", frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

pivot.round(3)


**Observation.** The ordering Kharif > Rabi > Zaid holds for all eight crops without exception.
The seasonal effect is therefore a property of the season, not of what is grown in it.


## 8. Q3 — What distinguishes the three growing environments?


In [ ]:
env = ["Rainfall_mm", "Avg_Temperature_C", "Humidity_pct", "Soil_Moisture_pct",
       "Sunlight_Hours_Day", "Disease_Pest_Risk_pct"]
labels = ["Rainfall (mm)", "Avg temp (C)", "Humidity (%)", "Soil moisture (%)",
          "Sunlight (hrs/day)", "Pest/disease risk (%)"]

fig, axes = plt.subplots(2, 3, figsize=(11, 5.6))
for a, col, title in zip(axes.ravel(), env, labels):
    v = df.groupby("Season", observed=True)[col].mean()[SEASON_ORDER]
    b = a.bar(SEASON_ORDER, v, color=[SEASON_COLORS[s] for s in SEASON_ORDER], width=.62)
    a.bar_label(b, fmt="%.1f", padding=2, fontsize=9, fontweight="bold")
    a.set_title(title, fontsize=11); a.set_ylim(0, v.max() * 1.28); a.set_yticks([])
fig.suptitle("Each season is a distinct growing environment", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

df.groupby("Season", observed=True)[env].mean().round(2)


**Observation.** Kharif is wet (852 mm rain, 31.2% soil moisture) but carries the highest
pest/disease risk (54.5%). Zaid is hot and dry (299 mm, 31.0 C) with the most sunlight, and the
lowest pest risk — heat and water scarcity, not pests, are what limit it.


## 9. Q4 — How do the economics move?


In [ ]:
econ = df.groupby("Season", observed=True)[
    ["Revenue_INR", "Total_Cost_INR", "Profit_INR", "Profit_per_ha"]].mean().round(0)
econ["loss_share_%"] = (df.groupby("Season", observed=True).Is_Loss.mean() * 100).round(1)
econ


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))

e = df.groupby("Season", observed=True)[["Revenue_INR", "Total_Cost_INR"]].mean().loc[SEASON_ORDER] / 1e5
x = np.arange(3)
ax[0].bar(x - .19, e.Revenue_INR, .38, label="Revenue", color=KHARIF)
ax[0].bar(x + .19, e.Total_Cost_INR, .38, label="Cost", color="#B8A88A")
ax[0].set_xticks(x); ax[0].set_xticklabels(SEASON_ORDER); ax[0].legend(frameon=False)
ax[0].set_title("Revenue falls, cost does not"); ax[0].set_ylabel("Rs lakh per farm (avg)")

sns.boxplot(data=df, x="Season", y="Profit_per_ha", order=SEASON_ORDER, ax=ax[1],
            palette=SEASON_COLORS, hue="Season", legend=False, showfliers=False, width=.55)
ax[1].axhline(0, color="#444", lw=1.2)
ax[1].set_title("Profit per hectare by season"); ax[1].set_ylabel("Rs per hectare"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()


**Observation.** This is the core economic finding. Average revenue falls from Rs 7.1 lakh (Kharif)
to Rs 5.2 lakh (Zaid), a drop of Rs 1.9 lakh, while average cost does **not** fall — it rises slightly
to Rs 5.4 lakh, because a dry season still demands irrigation, labour and inputs. Cost is
season-inelastic while revenue is season-sensitive, which is exactly why the median Zaid farm
slips into loss.


## 10. Q5 — Does irrigation method change the seasonal hit?


In [ ]:
yield_by_irr = df.pivot_table(index="Irrigation_Method", columns="Season",
                              values="Yield_Index", observed=True)[SEASON_ORDER]
yield_by_irr = yield_by_irr.loc[yield_by_irr.mean(axis=1).sort_values(ascending=False).index]

water_eff = df.pivot_table(index="Irrigation_Method", columns="Season",
                           values="Water_Efficiency_t_per_1000m3",
                           observed=True)[SEASON_ORDER].loc[yield_by_irr.index]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
yield_by_irr.plot(kind="bar", ax=ax[0], color=[KHARIF, RABI, ZAID], width=.78, rot=0)
ax[0].axhline(1, ls="--", lw=1, color="#888")
ax[0].set_title("Pressurised irrigation holds up best in Zaid")
ax[0].set_ylabel("Yield index"); ax[0].set_xlabel(""); ax[0].legend(frameon=False, ncol=3, fontsize=9)

sns.heatmap(water_eff, annot=True, fmt=".2f", cmap="YlGn", ax=ax[1],
            cbar=False, linewidths=1.5, linecolor="white")
ax[1].set_title("Water efficiency (t per 1000 m3)"); ax[1].set_ylabel(""); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()

print((yield_by_irr["Kharif"] - yield_by_irr["Zaid"]).round(3).rename("Kharif -> Zaid yield drop"))


**Observation.** Every method loses ground in Zaid, but the floor is much higher for pressurised
systems: sprinkler ends at a 0.89 yield index and drip at 0.84, against 0.73 for flood and 0.74 for
rainfed. Note the two different readings — drip starts highest (1.22 in Kharif) and therefore shows
the largest absolute drop (0.38), while sprinkler is the most *stable* (drop of 0.23); what matters
for a farmer sowing in Zaid is the level reached, and that is highest for sprinkler and drip. On
water productivity drip is the clear leader, roughly double flood irrigation (5.30 vs 2.73 t per
1000 m3 in Zaid). Rainfed's high efficiency ratio is an artefact of using very little purchased
water, not of producing more.


## 11. Q6 — What correlates with yield and profit?


In [ ]:
cols = ["Yield_Tonnes_Ha", "Profit_per_ha", "Water_Efficiency_t_per_1000m3", "Market_Price_INR_Tonne",
        "Rainfall_mm", "Soil_Moisture_pct", "Avg_Temperature_C", "Fertilizer_kg_ha",
        "Seed_Quality_Score", "Disease_Pest_Risk_pct"]

plt.figure(figsize=(8.2, 6.4))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="BrBG", center=0, vmin=-1, vmax=1,
            linewidths=1, linecolor="white", annot_kws={"size": 8}, cbar_kws={"shrink": .8})
plt.title("Correlation matrix: outcomes vs conditions", pad=12)
plt.xticks(rotation=40, ha="right", fontsize=8.5); plt.yticks(fontsize=8.5)
plt.tight_layout(); plt.show()


In [ ]:
# Within-season correlation of rainfall with yield — does more rain help inside a season?
df.groupby("Season", observed=True).apply(
    lambda d: d[["Rainfall_mm", "Yield_Tonnes_Ha"]].corr().iloc[0, 1], include_groups=False
).round(3)


**Observation.**
- Water efficiency and yield move together almost perfectly (r = 0.92), and profit per hectare
  follows yield (r = 0.54) — output, not price, is the main lever.
- Market price correlates *negatively* with yield (r = -0.38): high-value crops such as chilli are
  low-tonnage, so tonnes and rupees per tonne trade off.
- Fertiliser, pesticide and seed-quality scores show essentially no linear relationship with yield.
  Within this dataset, spending more on inputs does not buy yield.
- Rainfall correlates with yield **between** seasons but not **within** a season (r = -0.02 to 0.05).
  What matters is the seasonal regime as a whole, not a few extra millimetres in a given year.


## 12. Q7 — Is the pattern uniform across states?


In [ ]:
state_profit = df.pivot_table(index="State", columns="Season",
                              values="Profit_per_ha", observed=True)[SEASON_ORDER]
state_profit = state_profit.loc[state_profit.Kharif.sort_values().index] / 1000

ax = state_profit.plot(kind="barh", figsize=(9, 5), color=[KHARIF, RABI, ZAID], width=.78)
ax.axvline(0, color="#444", lw=1.2)
ax.set_title("Punjab and Karnataka stay profitable in Zaid; the rest do not")
ax.set_xlabel("Profit per hectare (Rs thousand)"); ax.set_ylabel("")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

state_profit.round(1)


**Observation.** The seasonal direction is universal, but the severity is not. Punjab
(+Rs 6,358/ha) and Karnataka (+Rs 4,903/ha) remain profitable in Zaid, while Andhra Pradesh
(-Rs 8,455/ha) and Gujarat (-Rs 7,456/ha) fall deepest. Punjab is also the only state where Rabi
beats Kharif — consistent with its wheat-centred, assured-irrigation system.


## 13. Conclusions

1. **Seasonal performance is ordered and consistent:** Kharif > Rabi > Zaid on yield (index 1.11 /
   0.96 / 0.79), profitability and water efficiency. Kruskal-Wallis and ANOVA both confirm the
   differences at p < 0.001.
2. **The gap is not a crop-mix artefact.** Crop composition is near-identical across seasons and
   Kharif leads within all eight crops individually.
3. **Cost is season-inelastic, revenue is not.** Revenue falls Rs 1.9 lakh from Kharif to Zaid while
   cost rises slightly — which is why 64.5% of Zaid farms are loss-making against 42.2% in Kharif.
4. **Water productivity is the strongest correlate of yield** (r = 0.92), far ahead of fertiliser,
   pesticide or seed-quality score, none of which show a meaningful linear effect.
5. **Irrigation method decides the severity of the dry season.** Sprinkler (0.89) and drip (0.84)
   hold a far higher Zaid yield index than flood (0.73) or rainfed (0.74), and drip roughly doubles
   flood irrigation's water productivity.
6. **Kharif's weak point is biological, not climatic:** pest/disease risk peaks at 54.5%, versus
   38.2% in Zaid.
7. **Regional response varies:** Punjab and Karnataka stay profitable in Zaid; Andhra Pradesh and
   Gujarat do not.

## 14. Recommendations

| # | Recommendation | Evidence |
|---|---|---|
| 1 | Treat Zaid as an **optional, cost-controlled** season rather than a default third crop; sow only where irrigation is assured | 64.5% of Zaid farms loss-making; median profit/ha negative |
| 2 | Prioritise **pressurised irrigation (drip / sprinkler) for Zaid cultivation**, targeting flood-irrigated farms first | Zaid yield index 0.89 sprinkler and 0.84 drip vs 0.73 flood; drip 5.30 vs flood 2.73 t per 1000 m3 |
| 3 | Shift the policy lever from input subsidy to **water-use efficiency**, since input intensity shows no yield payoff | r(fertiliser, yield) ~ 0.00; r(water efficiency, yield) = 0.92 |
| 4 | Fund **Kharif pest surveillance and IPM advisories** — the high-yield season's main downside risk | Pest/disease risk 54.5% in Kharif |
| 5 | Make advisories **state-specific**, using Punjab's and Karnataka's Zaid practice as a model for Andhra Pradesh and Gujarat | Zaid profit/ha ranges +Rs 6,358 to -Rs 8,455 across states |
| 6 | Push **cost flexibility** — shorter-duration varieties, shared machinery, staged input purchase — so cost can fall with revenue in weaker seasons | Cost in Zaid (Rs 5.4 L) exceeds Kharif (Rs 5.3 L) despite lower output |

## 15. Limitations
- The dataset covers a single period, so a seasonal effect cannot be separated from a year-specific
  weather shock.
- All findings are associational; no causal or predictive model is fitted.
- Prices are recorded per farm as a single figure, so within-season price movement is invisible.
- Missing values (~3% of three columns) were imputed by group median, which slightly reduces variance
  in those columns.
